# 04 · Avaliação — Baseline vs Fine-Tuned

Avalia os dois modelos com métricas padrão de IR usando `pytrec_eval`.

| Métrica | Descrição |
|---|---|
| MAP | Mean Average Precision |
| MRR | Mean Reciprocal Rank |
| nDCG@10 | Qualidade dos top-10 resultados |
| P@10 | Precisão nos 10 primeiros |
| Recall@100/1000 | Cobertura de relevantes |

## 1 · Instalação

In [2]:
!py -m pip install sentence-transformers faiss-cpu pytrec-eval-terrier

## 2 · Imports e configuração

In [7]:
import json, pickle
from pathlib import Path
import faiss, numpy as np, pytrec_eval
from sentence_transformers import SentenceTransformer

FINETUNED_MODEL = 'models/finetuned'
TOP_K           = 1000
BATCH_SIZE      = 64
DATA_DIR        = Path('data')
RESULTS_DIR     = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

METRICS = {'map', 'P_1', 'P_5', 'P_10', 'ndcg_cut_10', 'ndcg_cut_100', 'recip_rank',
           'recall_100', 'recall_1000'}

## 3 · Funções auxiliares

In [4]:
def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def load_qrels(path):
    qrels = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 4:
                qid, _, did, rel = parts
                qrels.setdefault(qid, {})[did] = int(rel)
    return qrels

def load_run(path):
    run = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 6:
                qid, _, did, _, score, _ = parts
                run.setdefault(qid, {})[did] = float(score)
    return run

## 4 · Geração do run fine-tuned

> Se `run_finetuned.txt` já existir, esta célula pula a geração.

In [5]:
corpus  = load_jsonl(DATA_DIR / 'corpus.jsonl')
queries = load_jsonl(DATA_DIR / 'queries.jsonl')
RUN_FT  = RESULTS_DIR / 'run_finetuned.txt'

if RUN_FT.exists():
    print(f'[skip] {RUN_FT} já existe.')
else:
    print(f'Carregando modelo fine-tuned: {FINETUNED_MODEL}')
    model = SentenceTransformer(FINETUNED_MODEL)

    # Embeddings do corpus
    print('Gerando embeddings do corpus...')
    doc_texts = [d['text']   for d in corpus]
    doc_ids   = [d['doc_id'] for d in corpus]
    doc_embs  = model.encode(doc_texts, batch_size=BATCH_SIZE,
                             show_progress_bar=True,
                             normalize_embeddings=True,
                             convert_to_numpy=True).astype('float32')

    index = faiss.IndexFlatIP(doc_embs.shape[1])
    index.add(doc_embs)

    # Embeddings das queries
    print('Gerando embeddings das queries...')
    q_texts = [q['text'] for q in queries]
    q_embs  = model.encode(q_texts, batch_size=BATCH_SIZE,
                           show_progress_bar=True,
                           normalize_embeddings=True,
                           convert_to_numpy=True).astype('float32')

    scores, idxs = index.search(q_embs, TOP_K)

    with open(RUN_FT, 'w') as f:
        for i, q in enumerate(queries):
            for rank, (idx, sc) in enumerate(zip(idxs[i], scores[i]), 1):
                f.write(f'{q["query_id"]}\tQ0\t{doc_ids[idx]}\t{rank}\t{sc:.6f}\tfinetuned\n')

    print(f'Run salva → {RUN_FT}')

Carregando modelo fine-tuned: models/finetuned


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Gerando embeddings do corpus...


Batches:   0%|          | 0/5777 [00:00<?, ?it/s]

Gerando embeddings das queries...


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Run salva → results\run_finetuned.txt


## 5 · Avaliação com pytrec_eval

In [8]:
qrels = load_qrels(DATA_DIR / 'qrels.txt')

run_b  = load_run(RESULTS_DIR / 'run_baseline.txt')
run_ft = load_run(RESULTS_DIR / 'run_finetuned.txt')

def evaluate(run, qrels, metrics):
    ev = pytrec_eval.RelevanceEvaluator(qrels, metrics)
    pq = ev.evaluate(run)
    avg = {m: np.mean([r[m] for r in pq.values() if m in r]) for m in metrics}
    return avg, pq

b_avg,  b_pq  = evaluate(run_b,  qrels, METRICS)
ft_avg, ft_pq = evaluate(run_ft, qrels, METRICS)
print('Avaliação concluída!')

Avaliação concluída!


## 6 · Tabela comparativa

In [9]:
DISPLAY = {k: v for k, v in {
    'map':          'MAP',
    'recip_rank':   'MRR',
    'ndcg_cut_10':  'nDCG@10',
    'ndcg_cut_100': 'nDCG@100',
    'P_1':          'P@1',
    'P_5':          'P@5',
    'P_10':         'P@10',
    'recall_100':   'Recall@100',
    'recall_1000':  'Recall@1000',
}.items() if k in METRICS}

print(f'  {"Métrica":<15} {"Baseline":>10} {"Fine-tuned":>12} {"Δ":>8}  {"Melhora":>8}')
print('=' * 60)
for key, label in DISPLAY.items():
    b  = b_avg.get(key, 0)
    ft = ft_avg.get(key, 0)
    d  = ft - b
    pct = (d / b * 100) if b > 0 else 0
    arrow = '▲' if d > 0 else ('▼' if d < 0 else '–')
    print(f'  {label:<15} {b:>10.4f} {ft:>12.4f} {arrow}{abs(d):>6.4f}  {pct:>+7.1f}%')

  Métrica           Baseline   Fine-tuned        Δ   Melhora
  MAP                 0.1560       0.1936 ▲0.0376    +24.1%
  MRR                 0.7254       0.5859 ▼0.1395    -19.2%
  nDCG@10             0.3724       0.3192 ▼0.0531    -14.3%
  nDCG@100            0.3794       0.3848 ▲0.0055     +1.4%
  P@1                 0.6295       0.4377 ▼0.1918    -30.5%
  P@5                 0.2895       0.3001 ▲0.0107     +3.7%
  P@10                0.1973       0.2328 ▲0.0355    +18.0%
  Recall@100          0.3231       0.4519 ▲0.1289    +39.9%
  Recall@1000         0.5172       0.7053 ▲0.1881    +36.4%


## 7 · Exportar resultados para CSV

In [10]:
import csv

csv_path = RESULTS_DIR / 'comparison.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['metric', 'baseline', 'finetuned', 'delta', 'pct_change'])
    for key in DISPLAY:
        b = b_avg.get(key, 0); ft = ft_avg.get(key, 0); d = ft - b
        pct = (d / b * 100) if b > 0 else 0
        w.writerow([key, f'{b:.6f}', f'{ft:.6f}', f'{d:.6f}', f'{pct:.2f}'])

print(f'CSV salvo → {csv_path}')

CSV salvo → results\comparison.csv
